# LAB | Intro to Machine Learning

**Load the data**

In this challenge, we will be working with Spaceship Titanic data. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [1]:
#import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [3]:
# Check the number of rows and columns
print("Shape of DataFrame:", spaceship.shape)  # `.shape` returns a (rows, columns) tuple :contentReference[oaicite:5]{index=5}


Shape of DataFrame: (8693, 14)


**Check for data types**

In [4]:
# Display each column name with its inferred dtype and non-null count
print(spaceship.info())  # `.info()` shows dtypes (object, float64, bool, etc.) and missing counts :contentReference[oaicite:7]{index=7}


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB
None


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check for missing values**

In [5]:
# Count of missing values in each column
print("Missing values per column:\n", spaceship.isnull().sum())  # `.isnull().sum()` tallies NaNs in each column :contentReference[oaicite:9]{index=9}


Missing values per column:
 PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [6]:
# Remove all rows with at least one missing value
spaceship_dropna = spaceship.dropna(axis=0, how="any")  # `dropna(how="any")` drops rows with any NaN :contentReference[oaicite:11]{index=11}

# Check the new shape to see how many rows were removed
print("New shape after dropna:", spaceship_dropna.shape)  # Confirms that rows with NaNs are gone :contentReference[oaicite:12]{index=12}


New shape after dropna: (6606, 14)


**KNN**

K Nearest Neighbors is a distance based algorithm, and requeries all **input data to be numerical.**

Let's only select numerical columns as our features.

In [7]:
# KNN requires numeric inputs; select only columns of numeric dtype
X = spaceship_dropna.select_dtypes(include=[np.number])  # `select_dtypes(include=[np.number])` filters numeric columns :contentReference[oaicite:14]{index=14}

print("Numeric feature columns:", X.columns.tolist())  # Show which numeric columns remain :contentReference[oaicite:15]{index=15}


Numeric feature columns: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']


And also lets define our target.

In [8]:
# Our target (y) is the boolean `Transported` column, converted to integer (0,1)
y = spaceship_dropna["Transported"].astype(int)  # Convert True/False to 1/0 for classification :contentReference[oaicite:17]{index=17}

print("Unique values in target y:", np.unique(y))  # Confirms that y contains [0, 1] :contentReference[oaicite:18]{index=18}


Unique values in target y: [0 1]


**Train Test Split**

Now that we have split the data into **features** and **target** variables and imported the **train_test_split** function, split X and y into X_train, X_test, y_train, and y_test. 80% of the data should be in the training set and 20% in the test set.

In [9]:
# Split X and y into training and test sets, with 80/20 split and a fixed random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)  # `stratify=y` keeps the same target distribution in train and test 

print("X_train shape:", X_train.shape, "| X_test shape:", X_test.shape)  # Verify dimensions after split :contentReference[oaicite:21]{index=21}
print("y_train distribution:", np.bincount(y_train), "| y_test distribution:", np.bincount(y_test))  # Verify class balance 


X_train shape: (5284, 6) | X_test shape: (1322, 6)
y_train distribution: [2623 2661] | y_test distribution: [656 666]


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

You need to choose between **Classificator** or **Regressor**. Take into consideration target variable to decide.

Initialize a KNN instance without setting any hyperparameter.

In [10]:
from sklearn.neighbors import KNeighborsClassifier  # KNN classifier for binary/multiclass problems 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # Metrics for classification performance 

# Initialize KNN with default hyperparameters (n_neighbors=5 by default)
knn = KNeighborsClassifier()  # No need to set n_neighbors manually for this exercise 

# Fit the KNN model on the training data
knn.fit(X_train, y_train)  # KNN learns training instances and their labels 

# Predict on the test set
y_pred = knn.predict(X_test)  # For each X_test sample, finds k nearest neighbors to vote on class 

# Evaluate model performance
acc = accuracy_score(y_test, y_pred)  # Overall accuracy (correct predictions / total predictions) 
print("Test Accuracy:", acc)

print("\nClassification Report:\n", classification_report(y_test, y_pred))  # Precision, recall, F1-score per class 

cm = confusion_matrix(y_test, y_pred)  # 2×2 confusion matrix for binary classification 
print("\nConfusion Matrix:\n", cm)


Test Accuracy: 0.75642965204236

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.72      0.75       656
           1       0.74      0.79      0.77       666

    accuracy                           0.76      1322
   macro avg       0.76      0.76      0.76      1322
weighted avg       0.76      0.76      0.76      1322


Confusion Matrix:
 [[473 183]
 [139 527]]


Fit the model to your data.

Evaluate your model.

**Congratulations, you have just developed your first Machine Learning model!**